# Fine-tuning Whisper (small) pour le fulfulde (fuv) — Adamaoua, Cameroun

Ce notebook :
1. Monte ton Google Drive
2. Clone/utilise le code du repo `fuv-stt-whisper`
3. Installe les dependances
4. Verifie ton dataset (audio + `metadata.json`)
5. Lance le fine-tuning de `openai/whisper-small`
6. Teste le modele obtenu sur un nouvel audio

Avant de commencer : pousse ton dossier de dataset sur ton Drive avec cette structure :

```
MyDrive/fuv-stt-dataset/
├── audio/
│   ├── 0001.wav
│   ├── 0002.wav
│   └── ...
└── metadata.json
```

Voir le README du repo pour le format exact de `metadata.json`.

## 1. Monter Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Récupérer le code du projet\n\nRemplace l'URL par celle de ton repo GitHub une fois que tu l'auras créé et poussé.

In [3]:
%cd /content
!git clone https://github.com/bono-p/fuv-stt-whisper.git
%cd fuv-stt-whisper

/content
Cloning into 'fuv-stt-whisper'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 42 (delta 15), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (42/42), 25.21 KiB | 8.40 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/fuv-stt-whisper


## 3. Installer les dépendances

In [4]:
!pip install -r requirements.txt --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.6 MB/s eta 0:00:00


## 3bis. Purifier les audios (.m4a → .wav 16kHz mono)\n\nSi tes enregistrements sont en `.m4a`, cette étape les convertit au format attendu par Whisper (`.wav` mono 16kHz), met à jour `metadata.json` automatiquement, et conserve les `.m4a` originaux dans un sous-dossier `audio/m4a_originaux/` (rien n'est supprimé). `ffmpeg` est déjà installé sur Colab, aucune installation supplémentaire nécessaire.\n\nSi tes fichiers sont déjà en `.wav`, cette cellule ne fait rien (message 'aucun fichier .m4a trouve').

In [5]:
%cd src
!python convert_audio.py "/content/drive/MyDrive/fuv-stt-dataset/audio" "/content/drive/MyDrive/fuv-stt-dataset/audio"
%cd ..

/content/fuv-stt-whisper/src
✅ ffmpeg trouvé : ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers

🎵 Conversion audio — Projet Tardigrade
────────────────────────────────────────────────────
   Source    : /content/drive/MyDrive/fuv-stt-dataset/audio
   Sortie    : /content/drive/MyDrive/fuv-stt-dataset/audio
   Fichiers  : 972 fichier(s) '.m4a'
   Format    : 16000 Hz · Mono · WAV 16-bit
   Renommage : non (nom d'origine)

  [████████████████████████████████████████] 972/972  100.0%                                

════════════════════════════════════════════════════
  RAPPORT DE CONVERSION
────────────────────────────────────────────────────
  Fichiers traités   : 972
  Convertis avec succès : 972
  Format de sortie   : WAV · 16000 Hz · Mono · PCM 16-bit
  Dossier de sortie  : /content/drive/MyDrive/fuv-stt-dataset/audio
  Durée totale estimée : 43.2 min (2589s)
════════════════════════════════════════════════════
/content/fuv-stt-whisper


## 4. Vérifier / ajuster la configuration\n\nOuvre `src/config.py` si besoin (chemin Drive, taille du modele, hyperparametres). Par defaut tout pointe vers `/content/drive/MyDrive/fuv-stt-dataset`.

In [6]:
!cat src/config.py

"""
Configuration centrale du pipeline de fine-tuning Whisper pour le fulfulde (fuv).

Toutes les valeurs sont modifiables ici sans toucher au reste du code.
"""

from dataclasses import dataclass, field


@dataclass
class DataConfig:
    # Dossier racine sur ton Google Drive contenant :
    #   - un sous-dossier "audio/" avec 0001.wav, 0002.wav, ...
    #   - un fichier metadata.json (voir format dans le README)
    drive_root: str = "/content/drive/MyDrive/fuv-stt-dataset"
    audio_subdir: str = "audio"
    metadata_filename: str = "metadata.json"

    # Proportion des donnees reservee a la validation (le reste = train)
    eval_split_ratio: float = 0.1

    # Frequence d'echantillonnage attendue par Whisper
    sampling_rate: int = 16000

    # Seed pour que le split train/val soit reproductible
    seed: int = 42


@dataclass
class ModelConfig:
    # Checkpoint de base. "openai/whisper-small" = bon compromis
    # qualite/vitesse pour un dataset de quelques heures sur GPU Colab (T

## 5. Vérification rapide du dataset (avant d'entraîner)\n\nCharge juste les métadonnées et vérifie que tous les fichiers audio existent, sans lancer l'entraînement complet.

In [7]:
%cd src
from dataset import load_raw_dataset
raw = load_raw_dataset()
print(raw)
print("\nExemple :", raw["train"][0]["text"])

/content/fuv-stt-whisper/src
[OK] 965 paires audio/texte valides chargees.
[OK] Split effectue -> train: 868 | eval: 97
DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 868
    })
    eval: Dataset({
        features: ['audio', 'text'],
        num_rows: 97
    })
})

Exemple : En fuu en keɓi geɗal meeɗen


## 6. Lancer le fine-tuning\n\nSelon la taille de ton dataset (~2h d'audio) et le GPU Colab attribué, compte entre 30 minutes et 2 heures pour `max_steps=1000`.

In [8]:
!python train.py

=== 1/4 : Chargement du modele et du processor ===
preprocessor_config.json: 100% 185k/185k [00:00<00:00, 189MB/s]
tokenizer_config.json: 100% 283k/283k [00:00<00:00, 212MB/s]
vocab.json: 100% 836k/836k [00:00<00:00, 33.3MB/s]
tokenizer.json: 100% 2.48M/2.48M [00:00<00:00, 143MB/s]
merges.txt: 100% 494k/494k [00:00<00:00, 96.9MB/s]
normalizer.json: 100% 52.7k/52.7k [00:00<00:00, 84.7MB/s]
added_tokens.json: 100% 34.6k/34.6k [00:00<00:00, 75.9MB/s]
special_tokens_map.json: 100% 2.19k/2.19k [00:00<00:00, 5.37MB/s]
config.json: 100% 1.97k/1.97k [00:00<00:00, 7.26MB/s]

model.safetensors: downloading bytes:  15% 142M/967M [00:01<00:05, 153MB/s, 12.4MB/s  ]
model.safetensors: downloading bytes:  17% 167M/967M [00:01<00:05, 141MB/s, 14.6MB/s  ]
model.safetensors: downloading bytes:  22% 217M/967M [00:02<00:04, 154MB/s, 17.8MB/s  ]
model.safetensors: downloading bytes:  49% 475M/967M [00:03<00:02, 191MB/s, 38.6MB/s  ]
model.safetensors: reconstructing file:  38% 369M/967M [00:03<00:07, 84.4MB

In [9]:
!cat src/train.py

cat: src/train.py: No such file or directory


In [ ]:
"""
Fine-tuning de Whisper (small par defaut) sur le dataset fulfulde (fuv).

Usage (depuis le notebook Colab, apres avoir monte Drive) :

    %cd /content/fuv-stt-whisper/src
    !python train.py

Le meilleur checkpoint (selon le WER de validation) est sauvegarde dans
train_config.output_dir, directement sur ton Google Drive pour survivre
au redemarrage de la VM Colab.
"""

import evaluate
import torch
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperFeatureExtractor,
    WhisperForConditionalGeneration,
    WhisperProcessor,
    WhisperTokenizer,
)

from config import data_config, model_config, train_config
from data_collator import DataCollatorSpeechSeq2SeqWithPadding
from dataset import build_processed_dataset

wer_metric = evaluate.load("wer")


def load_processor_and_model():
    feature_extractor = WhisperFeatureExtractor.from_pretrained(model_config.base_model)

    # NOTE : le fulfulde n'existe pas dans la liste des langues Whisper.
    # On utilise une langue "bootstrap" proche (cf config.py) uniquement
    # pour initialiser les tokens speciaux <|lang|> et <|task|>. Le
    # fine-tuning reapprend la vraie correspondance texte <-> audio,
    # independamment de ce choix.
    tokenizer = WhisperTokenizer.from_pretrained(
        model_config.base_model,
        language=model_config.bootstrap_language,
        task=model_config.task,
    )

    processor = WhisperProcessor.from_pretrained(
        model_config.base_model,
        language=model_config.bootstrap_language,
        task=model_config.task,
    )

    model = WhisperForConditionalGeneration.from_pretrained(model_config.base_model)

    # Desactive la contrainte de langue forcee : on laisse le modele
    # generer librement, ce qui est important puisqu'on sort du cadre
    # des langues originales de Whisper.
    model.generation_config.language = None

    # On doit forcer le token de debut de sequence a celui de la langue
    # cible (ici, on utilise le token de la langue bootstrapsee pour
    # Whisper, ie. Swahili). Si on n'appelle pas ca, le modele essaie
    # de generer un token <|en|>, ce qui ne nous interesse pas.
    model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(
        language=model_config.bootstrap_language,
        task=model_config.task,
    )

    # Force la generation a un seul token de langue au lieu de 2
    model.config.suppress_tokens = []

    return feature_extractor, tokenizer, processor, model


def compute_metrics(pred, tokenizer):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


def main():
    print("\n=== 1/4 : Chargement du modele et du processor ===")
    feature_extractor, tokenizer, processor, model = load_processor_and_model()

    print("\n=== 2/4 : Chargement et preparation du dataset ===")
    dataset = build_processed_dataset(feature_extractor, tokenizer, num_proc=1)

    print("\n=== 3/4 : Configuration de l'entrainement ===")
    data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

    use_fp16 = train_config.fp16 and torch.cuda.is_available()

    training_args = Seq2SeqTrainingArguments(
        output_dir=train_config.output_dir,
        per_device_train_batch_size=train_config.per_device_train_batch_size,
        per_device_eval_batch_size=train_config.per_device_eval_batch_size,
        gradient_accumulation_steps=train_config.gradient_accumulation_steps,
        learning_rate=train_config.learning_rate,
        warmup_steps=train_config.warmup_steps,
        max_steps=train_config.max_steps,
        eval_strategy=train_config.eval_strategy,
        eval_steps=train_config.eval_steps,
        save_steps=train_config.save_steps,
        save_total_limit=train_config.save_total_limit,
        logging_steps=train_config.logging_steps,
        fp16=use_fp16,
        predict_with_generate=True,
        generation_max_length=train_config.generation_max_length,
        load_best_model_at_end=train_config.load_best_model_at_end,
        metric_for_best_model=train_config.metric_for_best_model,
        greater_is_better=train_config.greater_is_better,
        push_to_hub=train_config.push_to_hub,
        hub_model_id=train_config.hub_model_id if train_config.push_to_hub else None,
        report_to=["none"],
    )

    trainer = Seq2SeqTrainer(
        args=training_args,
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["eval"],
        data_collator=data_collator,
        compute_metrics=lambda pred: compute_metrics(pred, tokenizer),
    )

    print("\n=== 4/4 : Entrainement ===")
    trainer.train()

    print("\nSauvegarde du modele final...")
    trainer.save_model(train_config.output_dir)
    processor.save_pretrained(train_config.output_dir)

    if train_config.push_to_hub:
        trainer.push_to_hub()

    print(f"\n[OK] Modele sauvegarde dans : {train_config.output_dir}")


if __name__ == "__main__":
    main()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")



=== 1/4 : Chargement du modele et du processor ===


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]


=== 2/4 : Chargement et preparation du dataset ===
[OK] 965 paires audio/texte valides chargees.
[OK] Split effectue -> train: 868 | eval: 97


Extraction des features audio + tokenisation du texte:   0%|          | 0/868 [00:00<?, ? examples/s]

Extraction des features audio + tokenisation du texte:   0%|          | 0/97 [00:00<?, ? examples/s]


=== 3/4 : Configuration de l'entrainement ===

=== 4/4 : Entrainement ===


Step,Training Loss,Validation Loss


## 7. Tester le modèle fine-tuné\n\nDépose un fichier audio de test (fulfulde, différent du dataset d'entraînement) et transcris-le.

In [ ]:
!python inference.py --audio /content/drive/MyDrive/fuv-stt-dataset/test_audio.wav

## 8. (Optionnel) Publier sur Hugging Face Hub\n\nPasse `push_to_hub = True` dans `config.py` avant l'entraînement, ou pousse manuellement le dossier `output_dir` a posteriori :

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

# Puis, si push_to_hub=False pendant l'entrainement :
# from transformers import WhisperForConditionalGeneration, WhisperProcessor
# from config import train_config
# model = WhisperForConditionalGeneration.from_pretrained(train_config.output_dir)
# processor = WhisperProcessor.from_pretrained(train_config.output_dir)
# model.push_to_hub("bonopassale/whisper-small-fuv")
# processor.push_to_hub("bonopassale/whisper-small-fuv")